In [3]:
import urllib.request
import os

os.makedirs('docs', exist_ok=True)

url = "https://www.plant-maintenance.com/articles/hydraulic_fluid_cleanliness.pdf"
output_path = "docs/maintenance_guide.pdf"

headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

req = urllib.request.Request(url, headers=headers)

with urllib.request.urlopen(req) as response:
    with open(output_path, 'wb') as f:
        f.write(response.read())

print("Done.")

Done.


# RAG Pipeline — Maintenance Document Q&A

**Goal:** Build a system that answers questions over maintenance documents
using retrieval-augmented generation.

**Pipeline:** Load docs → Chunk → Embed → Store in FAISS → Retrieve → Generate

**Why RAG:** The LLM has no knowledge of our specific equipment.
RAG grounds answers in our actual documentation, reducing hallucination
and making answers auditable, you can show the source passage.

In [6]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load all documents from the docs folder
docs_path = Path('docs')
documents = []

for file in docs_path.iterdir():
    if file.suffix == '.pdf':
        loader = PyPDFLoader(str(file))
        documents.extend(loader.load())
        print(f"Loaded PDF: {file.name}")
    elif file.suffix == '.txt':
        loader = TextLoader(str(file))
        documents.extend(loader.load())
        print(f"Loaded TXT: {file.name}")

print(f"\nTotal pages/sections loaded: {len(documents)}")
print(f"\nSample of first document:")
print(documents[0].page_content[:500])

Loaded PDF: maintenance_guide.pdf

Total pages/sections loaded: 5

Sample of first document:
Defining And Maintaining Fluid Cleanliness For Maximum Hydraulic Component Life 
 
By Brendan Casey 
 
Many factors can reduce the service life of hydraulic components. Contamination of hydraulic 
fluid by insoluble particles is one of these factors. To prevent particle contamination from 
cutting short component life, an appropriate fluid cleanliness level must first be defined and 
then maintained on a continuous basis.  
 
Particle Contamination And Its Consequences 
 
Particle contamination 


In [7]:
# First try — 500 word chunks with 50 word overlap
text_splitter = RecursiveCharacterTextSplitter (
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")
print(f"\nSample chunk:")
print(chunks[0].page_content)
print(f"\nChunk metadata: {chunks[0].metadata}")

Number of chunks: 34
Average chunk size: 417 chars

Sample chunk:
Defining And Maintaining Fluid Cleanliness For Maximum Hydraulic Component Life 
 
By Brendan Casey 
 
Many factors can reduce the service life of hydraulic components. Contamination of hydraulic 
fluid by insoluble particles is one of these factors. To prevent particle contamination from 
cutting short component life, an appropriate fluid cleanliness level must first be defined and 
then maintained on a continuous basis.  
 
Particle Contamination And Its Consequences

Chunk metadata: {'producer': 'Acrobat Distiller 5.0.5 (Windows)', 'creator': 'Acrobat PDFMaker 5.0 for Word', 'creationdate': '2003-02-25T12:35:13+08:00', 'author': 'Brendan Casey', 'moddate': '2003-02-25T12:36:08+08:00', 'title': '1', 'source': 'docs/maintenance_guide.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [8]:
# Try smaller chunks — 300 chars
splitter_small = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks_small = splitter_small.split_documents(documents)

# Try larger chunks — 800 chars
splitter_large = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=80)
chunks_large = splitter_large.split_documents(documents)

print(f"Small chunks (300): {len(chunks_small)} chunks")
print(f"Medium chunks (500): {len(chunks)} chunks")
print(f"Large chunks (800): {len(chunks_large)} chunks")

Small chunks (300): 55 chunks
Medium chunks (500): 34 chunks
Large chunks (800): 21 chunks
